<a href="https://colab.research.google.com/github/Fu-Pei-Yin/Deep-Generative-Mode/blob/week7/%E6%99%82%E9%96%93%E5%BA%8F%E5%88%97.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ========================================
# HW7: 時間序列預測 (LSTM vs Transformer)
# 可在 Google Colab 直接執行
# ========================================

!pip install yfinance ta matplotlib seaborn torch torchvision torchaudio -q

import yfinance as yf
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
import math
import datetime
from tqdm import tqdm
import ta

# -----------------------
# 設定
# -----------------------
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

ticker = "2330.TW"   # or 'AAPL'
start_date = "2020-01-01"
end_date = "2024-12-31"
lookback = 30
batch_size = 64
device = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------
# 資料下載與技術指標
# -----------------------
df = yf.download(ticker, start=start_date, end=end_date)
df = df[['Open', 'High', 'Low', 'Close', 'Volume']]

# 技術指標
df['SMA_5'] = ta.trend.sma_indicator(df['Close'], window=5)
df['SMA_20'] = ta.trend.sma_indicator(df['Close'], window=20)
df['RSI_14'] = ta.momentum.rsi(df['Close'], window=14)
macd = ta.trend.macd(df['Close'])
df['MACD'] = macd
df['MACD_SIGNAL'] = ta.trend.macd_signal(df['Close'])
df['MACD_HIST'] = ta.trend.macd_diff(df['Close'])

df.dropna(inplace=True)

# 目標：log return
df['logret'] = np.log(df['Close']).diff().shift(-1)
df.dropna(inplace=True)

# -----------------------
# 時間切分
# -----------------------
n = len(df)
train_end = int(n * 0.8)
val_end = int(n * 0.9)

train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]

features = [c for c in df.columns if c not in ['logret']]
target = 'logret'

scaler = StandardScaler()
scaler.fit(train_df[features])

def scale(df):
    scaled = scaler.transform(df[features])
    return scaled, df[target].values

X_train, y_train = scale(train_df)
X_val, y_val = scale(val_df)
X_test, y_test = scale(test_df)

# -----------------------
# 建立 Dataset
# -----------------------
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y, lookback):
        self.X, self.y, self.lookback = X, y, lookback

    def __len__(self):
        return len(self.X) - self.lookback

    def __getitem__(self, idx):
        return (
            torch.tensor(self.X[idx:idx+self.lookback], dtype=torch.float32),
            torch.tensor(self.y[idx+self.lookback], dtype=torch.float32)
        )

train_ds = TimeSeriesDataset(X_train, y_train, lookback)
val_ds = TimeSeriesDataset(X_val, y_val, lookback)
test_ds = TimeSeriesDataset(X_test, y_test, lookback)

train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=False)
val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_dl = DataLoader(test_ds, batch_size=1, shuffle=False)

input_dim = len(features)

# -----------------------
# 模型 1: LSTM
# -----------------------
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.norm = nn.LayerNorm(hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.norm(out[:, -1, :])
        out = self.drop(out)
        return self.fc(out).squeeze()

# -----------------------
# 模型 2: Transformer Encoder
# -----------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class TransformerModel(nn.Module):
    def __init__(self, input_dim, d_model=128, nhead=4, num_layers=2, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_enc = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=256, dropout=dropout)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, 1)
    def forward(self, x, return_attn=False):
        x = self.input_proj(x)
        x = self.pos_enc(x)
        attn_weights = None
        out = self.encoder(x)
        if return_attn:
            attn_weights = self.encoder.layers[-1].self_attn.attn_output_weights
        pooled = out.mean(dim=1)
        return self.fc(pooled).squeeze(), attn_weights

# -----------------------
# 訓練與評估函式
# -----------------------
def train_model(model, optimizer, loader, criterion):
    model.train()
    total = 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(y)
    return total / len(loader.dataset)

def eval_model(model, loader, criterion):
    model.eval()
    total = 0
    preds, trues = [], []
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            out = model(X)
            preds.extend(out.cpu().numpy())
            trues.extend(y.cpu().numpy())
            total += criterion(out, y).item() * len(y)
    return total / len(loader.dataset), np.array(preds), np.array(trues)

# -----------------------
# 模型訓練
# -----------------------
def run_experiment(model, optimizer, scheduler=None, epochs=30, name="Model"):
    criterion = nn.MSELoss()
    model.to(device)
    train_losses, val_losses = [], []
    for epoch in tqdm(range(epochs)):
        train_loss = train_model(model, optimizer, train_dl, criterion)
        val_loss, _, _ = eval_model(model, val_dl, criterion)
        if scheduler:
            scheduler.step()
        train_losses.append(train_loss)
        val_losses.append(val_loss)
    plt.plot(val_losses)
    plt.title(f"{name} Validation RMSE Curve")
    plt.xlabel("Epoch")
    plt.ylabel("MSE")
    plt.show()
    return model

# LSTM
lstm = LSTMModel(input_dim).to(device)
opt_lstm = optim.Adam(lstm.parameters(), lr=1e-4)
lstm = run_experiment(lstm, opt_lstm, name="LSTM")

# Transformer
trans = TransformerModel(input_dim).to(device)
opt_trans = optim.AdamW(trans.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(opt_trans, T_0=10)
trans = run_experiment(trans, opt_trans, scheduler, name="Transformer")

# -----------------------
# 評估與反標準化
# -----------------------
def evaluate(model, loader, name):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            pred, *_ = model(X) if isinstance(model, TransformerModel) else (model(X),)
            preds.append(pred.item())
            trues.append(y.item())
    preds, trues = np.array(preds), np.array(trues)

    mse = mean_squared_error(trues, preds)
    mae = mean_absolute_error(trues, preds)
    rmse = np.sqrt(mse)

    print(f"{name} - Standardized: MSE={mse:.6f}, MAE={mae:.6f}, RMSE={rmse:.6f}")

    # 還原為價格尺度
    close_train = train_df['Close'].values
    last_price = close_train[-1]
    pred_price = np.exp(np.cumsum(preds)) * last_price
    true_price = np.exp(np.cumsum(trues)) * last_price

    mse_p = mean_squared_error(true_price, pred_price)
    mae_p = mean_absolute_error(true_price, pred_price)
    rmse_p = np.sqrt(mse_p)
    print(f"{name} - Price Scale: MSE={mse_p:.2f}, MAE={mae_p:.2f}, RMSE={rmse_p:.2f}")

    plt.figure()
    plt.plot(true_price, label="True")
    plt.plot(pred_price, label="Pred")
    plt.title(f"{name} Prediction (Price Scale)")
    plt.legend()
    plt.show()

# 評估 LSTM
evaluate(lstm, test_dl, "LSTM")

# 評估 Transformer
evaluate(trans, test_dl, "Transformer")

# -----------------------
# Transformer Attention heatmap
# -----------------------
X_sample, _ = next(iter(test_dl))
trans.eval()
with torch.no_grad():
    _, attn = trans(X_sample.to(device), return_attn=True)

if attn is not None:
    attn = attn.cpu().numpy()[0]
    plt.figure(figsize=(6,5))
    sns.heatmap(attn, cmap="viridis")
    plt.title("Transformer Attention Heatmap")
    plt.show()


In [ ]:
# ==============================================================================
# 1. 環境設定與套件安裝
# ==============================================================================
# 在 Google Colab 中運行以下代碼
!pip install yfinance pandas_ta scikit-learn torch matplotlib

import yfinance as yf
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import pandas_ta as ta
import math
import os
import random
from tqdm.auto import tqdm

# 設置隨機種子以確保可重現性
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ==============================================================================
# 2. 資料獲取與特徵工程
# ==============================================================================
TICKER = '2330.TW' #
START_DATE = '2020-01-01'
END_DATE = '2024-12-31'
LOOKBACK = 30 #
TARGET_MODE = 'logret' #

print(f"Downloading data for {TICKER}...")
data = yf.download(TICKER, start=START_DATE, end=END_DATE)

# 欄位：Open, High, Low, Close, Volume
# 技術指標：SMA_5, SMA_20, RSI_14, MACD, MACD_SIGNAL, MACD_HIST
data.ta.sma(length=5, append=True)
data.ta.sma(length=20, append=True)
data.ta.rsi(length=14, append=True)
data.ta.macd(append=True)

# 檢查生成的指標欄位名稱 (pandas_ta 預設命名)
indicator_cols = [
    'SMA_5', 'SMA_20', 'RSI_14',
    'MACD_12_26_9', 'MACDh_12_26_9', 'MACDs_12_26_9'
]
# 將 pandas_ta 的欄位名稱統一為作業要求的簡稱
data.rename(columns={
    'MACD_12_26_9': 'MACD',
    'MACDh_12_26_9': 'MACD_HIST',
    'MACDs_12_26_9': 'MACD_SIGNAL',
    f'RSI_{14}': 'RSI_14',
    f'SMA_{5}': 'SMA_5',
    f'SMA_{20}': 'SMA_20'
}, inplace=True)


# 目標：logret = log(Close[t+1]) - log(Close[t])
data['logret'] = np.log(data['Close']).shift(-1) - np.log(data['Close'])

# 特徵欄位
base_features = ['Open', 'High', 'Low', 'Close', 'Volume']
tech_features = ['SMA_5', 'SMA_20', 'RSI_14', 'MACD', 'MACD_SIGNAL', 'MACD_HIST']
features_list = base_features + tech_features
# 移除包含 NaN 的行 (通常是技術指標計算的前幾行和 logret 的最後一行)
data.dropna(inplace=True)

# 確保 logret 是預測 t+1 的值，所以最後一天的 Close/logret 不用於訓練
df = data.iloc[:-1].copy()

# 檢查處理後的資料
print(f"Data shape after cleaning: {df.shape}")
print(f"Features used: {features_list}")
INPUT_DIM = len(features_list)
print(f"Input dimension: {INPUT_DIM}")

# ==============================================================================
# 3. 資料切分與標準化
# ==============================================================================
# 時間序 8/1/1 切分 (Train/Val/Test)
total_size = len(df)
train_size = int(total_size * 0.8)
val_size = int(total_size * 0.1)
test_size = total_size - train_size - val_size

train_df = df.iloc[:train_size]
val_df = df.iloc[train_size:train_size + val_size]
test_df = df.iloc[train_size + val_size:total_size]

# 原始價格 (Close) 用於反標準化和指標回報
test_original_close = test_df['Close'].values.copy()
# logret 的預測目標是 t+1 的值，但用於指標計算和繪圖的是預測值與 logret 本身
# 在這個架構中，模型直接預測 logret，所以原始尺度的指標需要還原為價格
# 為了後續價格還原，需要每個測試樣本的 Close[t]
test_close_for_price_restore = test_df['Close'].shift(1).iloc[LOOKBACK:].values
# 測試集的真實 logret 值
test_true_logret = test_df[TARGET_MODE].iloc[LOOKBACK:].values

print(f"Train/Val/Test size: {len(train_df)} / {len(val_df)} / {len(test_df)}")


# 標準化：StandardScaler (只用訓練集 fit)
scaler = StandardScaler()
# Fit only on training data features
scaler.fit(train_df[features_list])

# Transform all splits
train_df[features_list] = scaler.transform(train_df[features_list])
val_df[features_list] = scaler.transform(val_df[features_list])
test_df[features_list] = scaler.transform(test_df[features_list])


# 自定義 Dataset
class TimeSeriesDataset(Dataset):
    def __init__(self, dataframe, features, target, lookback):
        self.features = dataframe[features].values
        self.target = dataframe[target].values
        self.lookback = lookback

    def __len__(self):
        return len(self.features) - self.lookback

    def __getitem__(self, i):
        # 輸入：lookback 天 x 多變量特徵
        # x: [lookback, input_dim]
        # y: [1] (logret[t+1])
        x = self.features[i:i + self.lookback]
        y = self.target[i + self.lookback - 1] # 注意：logret 已經是 t+1 的值，取 lookback-1 對應的 logret
        return torch.FloatTensor(x), torch.FloatTensor([y])

# 建立 DataLoader
BATCH_SIZE = 64 #

train_dataset = TimeSeriesDataset(train_df, features_list, TARGET_MODE, LOOKBACK)
val_dataset = TimeSeriesDataset(val_df, features_list, TARGET_MODE, LOOKBACK)
test_dataset = TimeSeriesDataset(test_df, features_list, TARGET_MODE, LOOKBACK)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False) # 測試集 batch_size=1 以便繪圖

# ==============================================================================
# 4. 模型定義
# ==============================================================================

# --- LSTM Baseline ---
class LSTMBaseline(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, dropout=0.1): #
        super(LSTMBaseline, self).__init__()
        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            num_layers=1, #
            batch_first=True,
            dropout=dropout #
        )
        self.layer_norm = nn.LayerNorm(hidden_dim) #
        self.linear = nn.Linear(hidden_dim, 1) #

    def forward(self, x):
        # x shape: [batch_size, lookback, input_dim]
        lstm_out, _ = self.lstm(x)
        # 取序列的最後一個時間步輸出
        last_step_out = lstm_out[:, -1, :]

        # LayerNorm
        norm_out = self.layer_norm(last_step_out)

        # Linear → 1
        output = self.linear(norm_out)
        return output

# --- Transformer (Encoder-only) ---
class PositionalEncoding(nn.Module):
    # Sinusoidal Positional Encoding
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0) # shape: [1, max_len, d_model]
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: [batch_size, seq_len, d_model]
        # Positional Encoding 被點對點地加到輸入上
        # 由於 x 已經是 projected embedding，PE 直接加在上面
        return x + self.pe[:, :x.size(1), :]

class TransformerEncoderOnly(nn.Module):
    def __init__(self, input_dim, d_model=128, n_head=8, num_layers=2, ffn_dim=256, dropout=0.1):
        super(TransformerEncoderOnly, self).__init__()

        # 1. Input projection (Linear 到 d_model)
        self.input_projection = nn.Linear(input_dim, d_model)

        # 2. Sinusoidal Positional Encoding
        self.pos_encoder = PositionalEncoding(d_model, LOOKBACK)

        # 3. Encoder Layer × 2
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_head,
            dim_feedforward=ffn_dim, # FFN=256
            dropout=dropout, # dropout=0.1
            batch_first=True,
            norm_first=False
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 4. Pooling: mean pooling (對時間維平均後接 Linear → 1)
        self.linear_out = nn.Linear(d_model, 1)

        # 用於儲存注意力權重
        self.attention_weights = None

    def forward(self, x):
        # x shape: [batch_size, lookback, input_dim]

        # 1. Input projection
        x = self.input_projection(x) # [batch_size, lookback, d_model]

        # 2. Positional Encoding
        x = self.pos_encoder(x)

        # 3. Encoder Stack (2 layers)
        # Pytorch TransformerEncoder 的 forward 方法不會返回 attention weights。
        # 需要修改或使用自定義的 Encoder Block 來回傳。
        # 為了滿足「最後一層回傳注意力權重」的要求，我們將手動執行最後一層 Encoder。

        # 執行第一層 Encoder
        if self.transformer_encoder.num_layers > 1:
            for i in range(self.transformer_encoder.num_layers - 1):
                x = self.transformer_encoder.layers[i](x)

        # 執行最後一層 Encoder 並保存 Attention
        final_encoder_layer = self.transformer_encoder.layers[-1]

        # 需要手動呼叫 MultiheadAttention 模塊來獲取 attn_output_weights

        # LayerNorm Before Multi-Head Attention
        norm_x = final_encoder_layer.norm1(x)

        # Multi-Head Attention
        attn_output, attn_output_weights = final_encoder_layer.self_attn(
            norm_x, norm_x, norm_x, average_attn_weights=False)

        # 儲存 Attention heatmap (平均所有 heads 的權重)
        # attn_output_weights shape: [batch_size, query_len, key_len] (這裡 query_len=key_len=lookback)
        self.attention_weights = attn_output_weights.mean(dim=1).detach().cpu().numpy()

        # Residual connection
        x = x + final_encoder_layer.dropout1(attn_output)

        # LayerNorm After Attention + Before FFN (如果 norm_first=True)
        # 這裡我們使用 norm_first=False，所以 LayerNorm 在 Residual 之後
        x = final_encoder_layer.norm_first and x or final_encoder_layer.norm1(x)

        # Feed Forward Network (FFN=256)
        x = x + final_encoder_layer.dropout2(final_encoder_layer.linear2(final_encoder_layer.dropout(final_encoder_layer.activation(final_encoder_layer.linear1(final_encoder_layer.norm2(x)))))

        x = not final_encoder_layer.norm_first and final_encoder_layer.norm2(x) or x

        # 4. Mean Pooling
        pooled_output = x.mean(dim=1) # 對時間維平均

        # 5. Linear → 1
        output = self.linear_out(pooled_output)
        return output

# ==============================================================================
# 5. 訓練與評估
# ==============================================================================

def train_and_evaluate(model_name, model, train_loader, val_loader, optimizer, scheduler=None, epochs=30): #
    print(f"\n--- Training {model_name} ---")
    model.to(DEVICE)
    criterion = nn.MSELoss()
    best_val_rmse = float('inf')
    patience = 5
    patience_counter = 0
    history = {'train_loss': [], 'val_rmse': []}

    for epoch in tqdm(range(1, epochs + 1), desc=f"{model_name} Epochs"):
        # Training loop
        model.train()
        train_loss = 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            output = model(x)
            loss = criterion(output, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * x.size(0)

        avg_train_loss = train_loss / len(train_loader.dataset)

        # Validation loop
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                output = model(x)
                loss = criterion(output, y)
                val_loss += loss.item() * x.size(0)

        avg_val_mse = val_loss / len(val_loader.dataset)
        val_rmse = np.sqrt(avg_val_mse)

        # Scheduler step (CosineAnnealingWarmRestarts in general, but simplified)
        if scheduler:
            scheduler.step(epoch + len(train_loader) / len(train_loader.dataset))

        # Save history for learning curve
        history['train_loss'].append(avg_train_loss)
        history['val_rmse'].append(val_rmse)

        # Early Stopping check
        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            patience_counter = 0
            torch.save(model.state_dict(), f'best_{model_name}.pt')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    # Load best model
    model.load_state_dict(torch.load(f'best_{model_name}.pt'))
    return model, history

def evaluate_model(model, test_loader, original_close_prices):
    model.eval()
    standardized_preds = []
    attentions = []

    with torch.no_grad():
        for x, _ in test_loader:
            x = x.to(DEVICE)
            output = model(x)
            standardized_preds.extend(output.squeeze().tolist())

            # 儲存 Attention weights (僅限 Transformer)
            if isinstance(model, TransformerEncoderOnly):
                attentions.append(model.attention_weights)

    standardized_preds = np.array(standardized_preds)

    # 1. 指標 (標準化空間)
    true_std = test_true_logret # 在 logret 模式下，真實目標值已是標準化的
    mse_std = mean_squared_error(true_std, standardized_preds)
    mae_std = mean_absolute_error(true_std, standardized_preds)
    rmse_std = np.sqrt(mse_std)

    # 2. 指標 (原始價格尺度)
    # 還原 logret 預測值回價格
    # log(Close[t+1]/Close[t]) = logret => Close[t+1] = Close[t] * exp(logret)

    # original_close_prices 已經是 Close[t] (lookback後的)
    predicted_log_price = np.log(original_close_prices) + standardized_preds
    predicted_prices = np.exp(predicted_log_price)

    # 真實價格 (t+1)
    true_prices = original_close_prices * np.exp(test_true_logret)

    # 原始價格尺度指標
    mse_orig = mean_squared_error(true_prices, predicted_prices)
    mae_orig = mean_absolute_error(true_prices, predicted_prices)
    rmse_orig = np.sqrt(mse_orig)

    # 3. 視覺化 Attention heatmap (僅限 Transformer)
    if attentions:
        # 取任一樣本的 Attention weights
        # 注意: attentions[0] 是 [n_heads, lookback, lookback] (平均後的 shape 是 [lookback, lookback])
        # 我們在模型中已經平均並 detach/cpu/numpy，所以 attentions[0] shape: [lookback, lookback]
        attn_weights_sample = attentions[0]

        plt.figure(figsize=(8, 8))
        plt.imshow(attn_weights_sample, cmap='viridis', origin='upper')
        plt.colorbar(label='Attention Weight')
        plt.xlabel('Key/Time step')
        plt.ylabel('Query/Time step')
        plt.title('Transformer Last-Layer Attention (one sample)')
        plt.show()

    # 4. 視覺化 True vs Pred (原始價格尺度)
    plt.figure(figsize=(15, 6))
    plt.plot(true_prices, label='True', color='tab:blue')
    plt.plot(predicted_prices, label='Pred', color='tab:orange', alpha=0.7)
    plt.title(f'{model_name} Prediction — {TICKER} ({TARGET_MODE})')
    plt.xlabel('Time (test set index)')
    plt.ylabel('Close')
    plt.legend()
    plt.show()

    return {
        'std': {'MSE': mse_std, 'MAE': mae_std, 'RMSE': rmse_std},
        'original': {'MSE': mse_orig, 'MAE': mae_orig, 'RMSE': rmse_orig}
    }, true_prices, predicted_prices

def plot_learning_curve(history, model_name):
    """繪製驗證集學習曲線 (epoch vs val RMSE)"""
    plt.figure(figsize=(10, 5))
    plt.plot(history['val_rmse'], label='Validation RMSE')
    plt.title(f'{model_name} Validation Learning Curve (Epoch vs Val RMSE)')
    plt.xlabel('Epoch')
    plt.ylabel('Val RMSE (Standardized Space)')
    plt.legend()
    plt.grid(True)
    plt.show()

# --- LSTM 訓練設定與執行 ---
lstm_model = LSTMBaseline(INPUT_DIM).to(DEVICE)
lstm_optimizer = optim.Adam(lstm_model.parameters(), lr=1e-4) #
lstm_model, lstm_history = train_and_evaluate(
    'LSTM', lstm_model, train_loader, val_loader, lstm_optimizer, epochs=30
)
lstm_metrics, lstm_true, lstm_pred = evaluate_model(
    lstm_model, test_loader, test_close_for_price_restore
)
plot_learning_curve(lstm_history, 'LSTM')

# --- Transformer 訓練設定與執行 ---
d_model = 128
transformer_model = TransformerEncoderOnly(INPUT_DIM, d_model=d_model).to(DEVICE)
transformer_optimizer = optim.AdamW(
    transformer_model.parameters(), lr=3e-4, weight_decay=1e-4
) #

# 簡化版的 Warmup + CosineAnnealingWarmRestarts (PyTorch LRS)
# 由於 Warmup 較為複雜，這裡使用 PyTorch 標準的 CosineAnnealingWarmRestarts
# 並手動在訓練循環中加入一個簡單的線性 Warmup 邏輯（在 train_and_evaluate 函數外定義）
# 或者更簡潔地，直接使用 AdamW + CosineAnnealing (因為無法直接訪問 transformers 庫中的 WarmupScheduler)
T_0 = 10 # CosineAnnealingWarmRestarts 的週期
transformer_scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    transformer_optimizer, T_0=T_0, T_mult=2
)
# 注意：作業要求是 Warmup(10% epochs) + CosineAnnealingWarmRestarts
# 為了滿足要求，我們必須在訓練循環中手動處理 Warmup。

# 重新定義帶有 Warmup 的訓練函數 (為符合嚴格要求)
def train_and_evaluate_transformer(model, train_loader, val_loader, optimizer, epochs=30, warmup_percent=0.1):
    print(f"\n--- Training Transformer (with Warmup & CosineAnnealing) ---")
    model.to(DEVICE)
    criterion = nn.MSELoss()
    best_val_rmse = float('inf')
    patience = 5
    patience_counter = 0
    history = {'train_loss': [], 'val_rmse': []}

    total_steps = len(train_loader) * epochs
    warmup_steps = int(total_steps * warmup_percent)
    current_step = 0

    T_0 = 10
    T_mult = 2
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=T_0, T_mult=T_mult, eta_min=1e-7 # 確保最低學習率不是 0
    )

    base_lr = optimizer.param_groups[0]['lr']

    for epoch in tqdm(range(1, epochs + 1), desc="Transformer Epochs"):
        # Training loop
        model.train()
        train_loss = 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()

            # **手動 Warmup 實現**
            if current_step < warmup_steps:
                warmup_factor = current_step / warmup_steps
                lr = base_lr * warmup_factor
                for param_group in optimizer.param_groups:
                    param_group['lr'] = lr
            elif current_step == warmup_steps:
                 # 轉換到 CosineAnnealing 的起始學習率
                 for param_group in optimizer.param_groups:
                    param_group['lr'] = base_lr

            output = model(x)
            loss = criterion(output, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * x.size(0)
            current_step += 1

            # **CosineAnnealing step**
            if current_step >= warmup_steps:
                scheduler.step() # 每個 batch step

        avg_train_loss = train_loss / len(train_loader.dataset)

        # Validation loop
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                output = model(x)
                loss = criterion(output, y)
                val_loss += loss.item() * x.size(0)

        avg_val_mse = val_loss / len(val_loader.dataset)
        val_rmse = np.sqrt(avg_val_mse)

        # Save history for learning curve
        history['train_loss'].append(avg_train_loss)
        history['val_rmse'].append(val_rmse)

        # Early Stopping check
        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            patience_counter = 0
            torch.save(model.state_dict(), f'best_Transformer.pt')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    # Load best model
    model.load_state_dict(torch.load(f'best_Transformer.pt'))
    return model, history


transformer_model, transformer_history = train_and_evaluate_transformer(
    transformer_model, train_loader, val_loader, transformer_optimizer, epochs=30, warmup_percent=0.1
)

transformer_metrics, transformer_true, transformer_pred = evaluate_model(
    transformer_model, test_loader, test_close_for_price_restore
)
plot_learning_curve(transformer_history, 'Transformer')


# ==============================================================================
# 6. 輸出結果
# ==============================================================================

# 輸出指標表
print("\n" + "="*50)
print(f"SUMMARY")
print(f"TICKER: {TICKER}, Period: {START_DATE} ~ {END_DATE}, TARGET_MODE={TARGET_MODE}")
print(f"Lookback: {LOOKBACK}, Features: {INPUT_DIM}")
print(f"Train/Val/Test size (rows): {len(train_df)} / {len(val_df)} / {len(test_df)}")
print("="*50)

# LSTM 指標
print(f"LSTM TEST (std):      "
      f"MSE={lstm_metrics['std']['MSE']:.6f}  "
      f"MAE={lstm_metrics['std']['MAE']:.6f}  "
      f"RMSE={lstm_metrics['std']['RMSE']:.6f}")
print(f"LSTM TEST (original): "
      f"MSE={lstm_metrics['original']['MSE']:.4f}  "
      f"MAE={lstm_metrics['original']['MAE']:.4f}  "
      f"RMSE={lstm_metrics['original']['RMSE']:.4f}")

# Transformer 指標
print(f"TR TEST (std):        "
      f"MSE={transformer_metrics['std']['MSE']:.6f}  "
      f"MAE={transformer_metrics['std']['MAE']:.6f}  "
      f"RMSE={transformer_metrics['std']['RMSE']:.6f}")
print(f"TR TEST (original):   "
      f"MSE={transformer_metrics['original']['MSE']:.4f}  "
      f"MAE={transformer_metrics['original']['MAE']:.4f}  "
      f"RMSE={transformer_metrics['original']['RMSE']:.4f}")
print("="*50)

# 圖表已在 evaluate_model 和 plot_learning_curve 中生成

# 技術指標比較分析 (此部分需要手動在報告中完成，程式碼無法自動產生)
print("\n技術指標比較與分析 (需在報告中完成)：")
print("以 RMSE、MAE 或可視化結果比較四類指標（趨勢型、震盪型、波動型、量能型）的貢獻與差異，並分析它們在 LSTM 與 Transformer 模型中的實驗效果。")
print("這需要在實驗中替換不同的特徵組合來執行，本腳本使用預設全部特徵。")

In [ ]:
"""
利用 LSTM 與 Transformer 預測股價時間序列
完整實作程式 - 可在 Google Colab 運行
"""

# ========== 1. 安裝與導入套件 ==========
!pip install yfinance ta -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import ta
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

# 設定隨機種子
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# 設定裝置
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用裝置: {device}")

# ========== 2. 資料下載與技術指標計算 ==========
def download_and_prepare_data(ticker='2330.TW', start='2020-01-01', end='2024-12-31'):
    """下載股價資料並計算技術指標"""
    print(f"下載 {ticker} 資料...")
    df = yf.download(ticker, start=start, end=end, progress=False)

    # 基本欄位
    df = df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()

    # 計算技術指標
    print("計算技術指標...")

    # 趨勢型指標
    df['SMA_5'] = ta.trend.sma_indicator(df['Close'], window=5)
    df['SMA_20'] = ta.trend.sma_indicator(df['Close'], window=20)
    df['EMA_12'] = ta.trend.ema_indicator(df['Close'], window=12)
    df['EMA_26'] = ta.trend.ema_indicator(df['Close'], window=26)

    # MACD
    macd = ta.trend.MACD(df['Close'])
    df['MACD'] = macd.macd()
    df['MACD_SIGNAL'] = macd.macd_signal()
    df['MACD_HIST'] = macd.macd_diff()

    # 震盪型指標
    df['RSI_14'] = ta.momentum.rsi(df['Close'], window=14)
    df['STOCH'] = ta.momentum.stoch(df['High'], df['Low'], df['Close'])

    # 波動型指標
    bollinger = ta.volatility.BollingerBands(df['Close'])
    df['BB_HIGH'] = bollinger.bollinger_hband()
    df['BB_LOW'] = bollinger.bollinger_lband()
    df['BB_MID'] = bollinger.bollinger_mavg()
    df['ATR'] = ta.volatility.average_true_range(df['High'], df['Low'], df['Close'])

    # 量能型指標
    df['OBV'] = ta.volume.on_balance_volume(df['Close'], df['Volume'])
    df['VOL_SMA'] = df['Volume'].rolling(window=20).mean()

    # 計算 log return 作為目標
    df['logret'] = np.log(df['Close']).diff()

    # 移除 NaN
    df = df.dropna()

    print(f"資料形狀: {df.shape}")
    print(f"日期範圍: {df.index[0]} 到 {df.index[-1]}")

    return df

# ========== 3. 資料集類別 ==========
class StockDataset(Dataset):
    """股價時間序列資料集"""
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def create_sequences(data, target, lookback=30):
    """創建時間序列樣本"""
    X, y = [], []
    for i in range(len(data) - lookback):
        X.append(data[i:i+lookback])
        y.append(target[i+lookback])
    return np.array(X), np.array(y)

def prepare_data_splits(df, feature_cols, target_col='logret', lookback=30,
                       train_ratio=0.8, val_ratio=0.1):
    """準備訓練/驗證/測試集（時間序列切分）"""
    # 提取特徵和目標
    features = df[feature_cols].values
    target = df[target_col].values

    # 創建序列
    X, y = create_sequences(features, target, lookback)

    # 時間序列切分
    n = len(X)
    train_size = int(n * train_ratio)
    val_size = int(n * val_ratio)

    X_train = X[:train_size]
    y_train = y[:train_size]

    X_val = X[train_size:train_size+val_size]
    y_val = y[train_size:train_size+val_size]

    X_test = X[train_size+val_size:]
    y_test = y[train_size+val_size:]

    # 標準化（只用訓練集 fit）
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train.reshape(-1, X_train.shape[-1])).reshape(X_train.shape)
    X_val_scaled = scaler_X.transform(X_val.reshape(-1, X_val.shape[-1])).reshape(X_val.shape)
    X_test_scaled = scaler_X.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(X_test.shape)

    y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
    y_val_scaled = scaler_y.transform(y_val.reshape(-1, 1)).flatten()
    y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1)).flatten()

    print(f"訓練集: {X_train_scaled.shape}, 驗證集: {X_val_scaled.shape}, 測試集: {X_test_scaled.shape}")

    return (X_train_scaled, y_train_scaled, X_val_scaled, y_val_scaled,
            X_test_scaled, y_test_scaled, scaler_X, scaler_y, y_test)

# ========== 4. LSTM 模型 ==========
class LSTMModel(nn.Module):
    """LSTM 基線模型"""
    def __init__(self, input_size, hidden_size=128, dropout=0.1):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True, dropout=dropout)
        self.layer_norm = nn.LayerNorm(hidden_size)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        lstm_out, (h_n, c_n) = self.lstm(x)
        # 取最後時間步
        last_output = lstm_out[:, -1, :]
        normalized = self.layer_norm(last_output)
        output = self.fc(normalized)
        return output.squeeze(-1)

# ========== 5. Transformer 模型 ==========
class PositionalEncoding(nn.Module):
    """正弦位置編碼"""
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class TransformerModel(nn.Module):
    """Transformer Encoder-only 模型"""
    def __init__(self, input_size, d_model=128, nhead=8, num_layers=2,
                 dim_feedforward=256, dropout=0.1):
        super(TransformerModel, self).__init__()
        self.input_projection = nn.Linear(input_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, 1)
        self.last_attention = None

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        x = self.input_projection(x)
        x = self.pos_encoder(x)

        # 儲存最後一層注意力權重
        for i, layer in enumerate(self.transformer_encoder.layers):
            if i == len(self.transformer_encoder.layers) - 1:
                # 最後一層，提取注意力
                attn_output, attn_weights = layer.self_attn(x, x, x, need_weights=True, average_attn_weights=True)
                self.last_attention = attn_weights
                x = layer.norm1(x + layer.dropout1(attn_output))
                ff_output = layer.linear2(layer.dropout(layer.activation(layer.linear1(x))))
                x = layer.norm2(x + layer.dropout2(ff_output))
            else:
                x = layer(x)

        # Mean pooling
        x = x.mean(dim=1)
        output = self.fc(x)
        return output.squeeze(-1)

# ========== 6. 訓練函數 ==========
def train_model(model, train_loader, val_loader, epochs=30, lr=1e-4,
                model_type='lstm', warmup_epochs=3):
    """訓練模型"""
    criterion = nn.MSELoss()

    if model_type == 'transformer':
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        total_steps = epochs * len(train_loader)
        warmup_steps = warmup_epochs * len(train_loader)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=10, T_mult=2
        )
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        scheduler = None

    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        # 訓練階段
        model.train()
        train_loss = 0
        for batch_idx, (X_batch, y_batch) in enumerate(train_loader):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            # Warmup
            if model_type == 'transformer' and epoch * len(train_loader) + batch_idx < warmup_steps:
                lr_scale = min(1., float(epoch * len(train_loader) + batch_idx + 1) / warmup_steps)
                for pg in optimizer.param_groups:
                    pg['lr'] = lr * lr_scale

            optimizer.zero_grad()
            output = model(X_batch)
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)
        train_losses.append(train_loss)

        # 驗證階段
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                output = model(X_batch)
                loss = criterion(output, y_batch)
                val_loss += loss.item()

        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        if scheduler and model_type == 'transformer':
            scheduler.step()

        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}")

    return train_losses, val_losses

def evaluate_model(model, test_loader, scaler_y):
    """評估模型"""
    model.eval()
    predictions = []
    actuals = []

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            output = model(X_batch)
            predictions.extend(output.cpu().numpy())
            actuals.extend(y_batch.numpy())

    predictions = np.array(predictions)
    actuals = np.array(actuals)

    # 標準化空間的指標
    mse_scaled = mean_squared_error(actuals, predictions)
    mae_scaled = mean_absolute_error(actuals, predictions)
    rmse_scaled = np.sqrt(mse_scaled)

    # 反標準化到原始尺度
    predictions_original = scaler_y.inverse_transform(predictions.reshape(-1, 1)).flatten()
    actuals_original = scaler_y.inverse_transform(actuals.reshape(-1, 1)).flatten()

    mse_original = mean_squared_error(actuals_original, predictions_original)
    mae_original = mean_absolute_error(actuals_original, predictions_original)
    rmse_original = np.sqrt(mse_original)

    return {
        'scaled': {'MSE': mse_scaled, 'MAE': mae_scaled, 'RMSE': rmse_scaled},
        'original': {'MSE': mse_original, 'MAE': mae_original, 'RMSE': rmse_original},
        'predictions_scaled': predictions,
        'actuals_scaled': actuals,
        'predictions_original': predictions_original,
        'actuals_original': actuals_original
    }

# ========== 7. 視覺化函數 ==========
def plot_predictions(actuals, predictions, title, model_name):
    """繪製預測結果"""
    plt.figure(figsize=(14, 5))
    plt.plot(actuals, label='True', alpha=0.7, linewidth=2)
    plt.plot(predictions, label='Predicted', alpha=0.7, linewidth=2)
    plt.title(f'{title} - {model_name}', fontsize=14, fontweight='bold')
    plt.xlabel('Time Steps', fontsize=12)
    plt.ylabel('Log Return', fontsize=12)
    plt.legend(fontsize=11)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{model_name}_prediction.png', dpi=300, bbox_inches='tight')
    plt.show()

def plot_learning_curve(train_losses, val_losses, model_name):
    """繪製學習曲線"""
    plt.figure(figsize=(10, 5))
    epochs = range(1, len(train_losses) + 1)
    plt.plot(epochs, train_losses, label='Train RMSE', marker='o', markersize=4)
    plt.plot(epochs, val_losses, label='Val RMSE', marker='s', markersize=4)
    plt.title(f'Learning Curve - {model_name}', fontsize=14, fontweight='bold')
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Loss (RMSE)', fontsize=12)
    plt.legend(fontsize=11)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{model_name}_learning_curve.png', dpi=300, bbox_inches='tight')
    plt.show()

def plot_attention_heatmap(model, test_loader, sample_idx=0):
    """繪製注意力熱圖"""
    model.eval()
    with torch.no_grad():
        for i, (X_batch, _) in enumerate(test_loader):
            if i == sample_idx:
                X_batch = X_batch.to(device)
                _ = model(X_batch)
                attention = model.last_attention[0].cpu().numpy()

                plt.figure(figsize=(12, 10))
                sns.heatmap(attention, cmap='viridis', cbar_kws={'label': 'Attention Weight'})
                plt.title('Transformer Attention Heatmap (Last Layer)', fontsize=14, fontweight='bold')
                plt.xlabel('Key Position', fontsize=12)
                plt.ylabel('Query Position', fontsize=12)
                plt.tight_layout()
                plt.savefig('attention_heatmap.png', dpi=300, bbox_inches='tight')
                plt.show()
                break

# ========== 8. 主程式 ==========
def main():
    print("=" * 60)
    print("股價預測：LSTM vs Transformer")
    print("=" * 60)

    # 下載資料
    df = download_and_prepare_data('2330.TW', '2020-01-01', '2024-12-31')

    # 定義特徵欄位（包含四類技術指標）
    feature_cols = [
        'Open', 'High', 'Low', 'Close', 'Volume',
        # 趨勢型
        'SMA_5', 'SMA_20', 'EMA_12', 'EMA_26', 'MACD', 'MACD_SIGNAL', 'MACD_HIST',
        # 震盪型
        'RSI_14', 'STOCH',
        # 波動型
        'BB_HIGH', 'BB_LOW', 'BB_MID', 'ATR',
        # 量能型
        'OBV', 'VOL_SMA'
    ]

    # 準備資料
    lookback = 30
    (X_train, y_train, X_val, y_val, X_test, y_test,
     scaler_X, scaler_y, y_test_original) = prepare_data_splits(
        df, feature_cols, 'logret', lookback
    )

    # 創建 DataLoader
    batch_size = 64
    train_dataset = StockDataset(X_train, y_train)
    val_dataset = StockDataset(X_val, y_val)
    test_dataset = StockDataset(X_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)

    input_size = X_train.shape[-1]

    # ========== LSTM 模型 ==========
    print("\n" + "=" * 60)
    print("訓練 LSTM 模型")
    print("=" * 60)
    lstm_model = LSTMModel(input_size, hidden_size=128, dropout=0.1).to(device)
    lstm_train_losses, lstm_val_losses = train_model(
        lstm_model, train_loader, val_loader, epochs=30, lr=1e-4, model_type='lstm'
    )

    lstm_results = evaluate_model(lstm_model, test_loader, scaler_y)
    print("\nLSTM 測試結果:")
    print(f"標準化空間 - MSE: {lstm_results['scaled']['MSE']:.6f}, "
          f"MAE: {lstm_results['scaled']['MAE']:.6f}, "
          f"RMSE: {lstm_results['scaled']['RMSE']:.6f}")
    print(f"原始尺度 - MSE: {lstm_results['original']['MSE']:.6f}, "
          f"MAE: {lstm_results['original']['MAE']:.6f}, "
          f"RMSE: {lstm_results['original']['RMSE']:.6f}")

    # ========== Transformer 模型 ==========
    print("\n" + "=" * 60)
    print("訓練 Transformer 模型")
    print("=" * 60)
    transformer_model = TransformerModel(
        input_size, d_model=128, nhead=8, num_layers=2,
        dim_feedforward=256, dropout=0.1
    ).to(device)

    transformer_train_losses, transformer_val_losses = train_model(
        transformer_model, train_loader, val_loader, epochs=30,
        lr=3e-4, model_type='transformer', warmup_epochs=3
    )

    transformer_results = evaluate_model(transformer_model, test_loader, scaler_y)
    print("\nTransformer 測試結果:")
    print(f"標準化空間 - MSE: {transformer_results['scaled']['MSE']:.6f}, "
          f"MAE: {transformer_results['scaled']['MAE']:.6f}, "
          f"RMSE: {transformer_results['scaled']['RMSE']:.6f}")
    print(f"原始尺度 - MSE: {transformer_results['original']['MSE']:.6f}, "
          f"MAE: {transformer_results['original']['MAE']:.6f}, "
          f"RMSE: {transformer_results['original']['RMSE']:.6f}")

    # ========== 視覺化 ==========
    print("\n" + "=" * 60)
    print("生成視覺化圖表")
    print("=" * 60)

    # 預測結果對比
    plot_predictions(
        lstm_results['actuals_original'],
        lstm_results['predictions_original'],
        'True vs Predicted (Original Scale)', 'LSTM'
    )

    plot_predictions(
        transformer_results['actuals_original'],
        transformer_results['predictions_original'],
        'True vs Predicted (Original Scale)', 'Transformer'
    )

    # 學習曲線
    lstm_rmse_train = [np.sqrt(loss) for loss in lstm_train_losses]
    lstm_rmse_val = [np.sqrt(loss) for loss in lstm_val_losses]
    plot_learning_curve(lstm_rmse_train, lstm_rmse_val, 'LSTM')

    transformer_rmse_train = [np.sqrt(loss) for loss in transformer_train_losses]
    transformer_rmse_val = [np.sqrt(loss) for loss in transformer_val_losses]
    plot_learning_curve(transformer_rmse_train, transformer_rmse_val, 'Transformer')

    # 注意力熱圖
    plot_attention_heatmap(transformer_model, test_loader, sample_idx=0)

    # ========== 技術指標比較實驗 ==========
    print("\n" + "=" * 60)
    print("技術指標比較實驗")
    print("=" * 60)

    indicator_groups = {
        '趨勢型': ['SMA_5', 'SMA_20', 'EMA_12', 'EMA_26', 'MACD', 'MACD_SIGNAL', 'MACD_HIST'],
        '震盪型': ['RSI_14', 'STOCH'],
        '波動型': ['BB_HIGH', 'BB_LOW', 'BB_MID', 'ATR'],
        '量能型': ['OBV', 'VOL_SMA']
    }

    base_features = ['Open', 'High', 'Low', 'Close', 'Volume']
    indicator_results = {}

    for group_name, indicators in indicator_groups.items():
        print(f"\n測試 {group_name} 指標...")
        test_features = base_features + indicators

        (X_train_ind, y_train_ind, X_val_ind, y_val_ind, X_test_ind, y_test_ind,
         scaler_X_ind, scaler_y_ind, _) = prepare_data_splits(
            df, test_features, 'logret', lookback
        )

        train_dataset_ind = StockDataset(X_train_ind, y_train_ind)
        val_dataset_ind = StockDataset(X_val_ind, y_val_ind)
        test_dataset_ind = StockDataset(X_test_ind, y_test_ind)

        train_loader_ind = DataLoader(train_dataset_ind, batch_size=batch_size, shuffle=True)
        val_loader_ind = DataLoader(val_dataset_ind, batch_size=batch_size)
        test_loader_ind = DataLoader(test_dataset_ind, batch_size=batch_size)

        # 測試 Transformer
        model_ind = TransformerModel(
            len(test_features), d_model=128, nhead=8, num_layers=2,
            dim_feedforward=256, dropout=0.1
        ).to(device)

        _, _ = train_model(
            model_ind, train_loader_ind, val_loader_ind, epochs=20,
            lr=3e-4, model_type='transformer', warmup_epochs=2
        )

        results_ind = evaluate_model(model_ind, test_loader_ind, scaler_y_ind)
        indicator_results[group_name] = results_ind

        print(f"{group_name} - RMSE: {results_ind['original']['RMSE']:.6f}, "
              f"MAE: {results_ind['original']['MAE']:.6f}")

    # 繪製技術指標比較圖
    plt.figure(figsize=(12, 6))
    groups = list(indicator_results.keys())
    rmse_values = [indicator_results[g]['original']['RMSE'] for g in groups]
    mae_values = [indicator_results[g]['original']['MAE'] for g in groups]

    x = np.arange(len(groups))
    width = 0.35

    plt.bar(x - width/2, rmse_values, width, label='RMSE', alpha=0.8)
    plt.bar(x + width/2, mae_values, width, label='MAE', alpha=0.8)

    plt.xlabel('技術指標類型', fontsize=12)
    plt.ylabel('誤差值', fontsize=12)
    plt.title('四類技術指標在 Transformer 模型中的表現比較', fontsize=14, fontweight='bold')
    plt.xticks(x, groups, fontsize=11)
    plt.legend(fontsize=11)
    plt.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig('indicator_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

    print("\n" + "=" * 60)
    print("所有實驗完成！")
    print("=" * 60)

    # 返回結果供報告使用
    return {
        'lstm': lstm_results,
        'transformer': transformer_results,
        'indicators': indicator_results
    }

# 執行主程式
if __name__ == '__main__':
    results = main()

In [ ]:
# 安裝必要套件
!pip install yfinance
!pip install torch
!pip install scikit-learn
!pip install matplotlib
!pip install pandas_ta

import yfinance as yf
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import pandas_ta as ta
import random
from torch.utils.data import Dataset, DataLoader

# 固定隨機種子
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# 讀取 Yahoo Finance 資料
ticker = "2330.TW"
start_date = "2020-01-01"
end_date = "2024-12-31"
df = yf.download(ticker, start=start_date, end=end_date)

# 計算技術指標
df["SMA_5"] = ta.sma(df["Close"], length=5)
df["SMA_20"] = ta.sma(df["Close"], length=20)
df["RSI_14"] = ta.rsi(df["Close"], length=14)
macd = ta.macd(df["Close"])
df["MACD"] = macd["MACD_12_26_9"]
df["MACD_SIGNAL"] = macd["MACDs_12_26_9"]
df["MACD_HIST"] = macd["MACDh_12_26_9"]

# 刪除含NaN的rows，確保全齡數據
df.dropna(inplace=True)

# 新增目標log return target: logret = log(Close[t+1]) - log(Close[t])
df["logClose"] = np.log(df["Close"])
df["logClose_t1"] = df["logClose"].shift(-1)
df["logret"] = df["logClose_t1"] - df["logClose"]
df.dropna(inplace=True)  # 去除最後一筆目標NaN

# 選擇特徵欄位（含技術指標）
feature_cols = ["Open", "High", "Low", "Close", "Volume",
                "SMA_5", "SMA_20", "RSI_14", "MACD", "MACD_SIGNAL", "MACD_HIST"]

target_col = "logret"

# 將資料依時間序嚴格分割 8:1:1 train/val/test
total_len = len(df)
train_len = int(total_len * 0.8)
val_len = int(total_len * 0.1)
test_len = total_len - train_len - val_len

train_df = df.iloc[:train_len]
val_df = df.iloc[train_len:train_len+val_len]
test_df = df.iloc[train_len+val_len:]

# 標準化 (只fit訓練集)
scaler_x = StandardScaler()
scaler_y = StandardScaler()

X_train = scaler_x.fit_transform(train_df[feature_cols])
X_val = scaler_x.transform(val_df[feature_cols])
X_test = scaler_x.transform(test_df[feature_cols])

y_train = scaler_y.fit_transform(train_df[[target_col]])
y_val = scaler_y.transform(val_df[[target_col]])
y_test = scaler_y.transform(test_df[[target_col]])

# 設置參數
lookback = 30
batch_size = 64
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 時間序列窗口化資料集
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y, lookback):
        self.X = X
        self.y = y
        self.lookback = lookback

    def __len__(self):
        return len(self.X) - self.lookback

    def __getitem__(self, idx):
        x = self.X[idx:idx+self.lookback]
        y = self.y[idx+self.lookback]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

train_dataset = TimeSeriesDataset(X_train, y_train, lookback)
val_dataset = TimeSeriesDataset(X_val, y_val, lookback)
test_dataset = TimeSeriesDataset(X_test, y_test, lookback)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

#— Baseline LSTM 模型定義 —
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, dropout=0.1):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.layernorm = nn.LayerNorm(hidden_dim)
        self.linear = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        out, _ = self.lstm(x)  # out: (batch, seq_len, hidden_dim)
        out = out[:, -1, :]    # 取最後一個時間點輸出
        out = self.dropout(out)
        out = self.layernorm(out)
        out = self.linear(out)
        return out.squeeze(-1)

#— Transformer Encoder-only 模型定義 —
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)  # (max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)]
        return x

class TransformerEncoderModel(nn.Module):
    def __init__(self, input_dim, d_model=64, nhead=8, num_layers=2, dim_feedforward=256, dropout=0.1):
        super(TransformerEncoderModel, self).__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers)
        self.layernorm = nn.LayerNorm(d_model)
        self.pool = nn.AdaptiveAvgPool1d(1)  # mean pooling over sequence length
        self.fc = nn.Linear(d_model, 1)
        self.attn_weights = None  # 用於儲存最後一層注意力權重

    def forward(self, x):
        # x: (batch, seq_len, input_dim)
        x = self.input_proj(x)  # (batch, seq_len, d_model)
        x = self.pos_encoder(x)
        # Transformer Encoder
        output = self.transformer_encoder(x)  # (batch, seq_len, d_model)
        self.attn_weights = None
        # mean pooling over sequence length dimension
        # 調整形狀以符合pooling需求
        output = output.transpose(1, 2)      # (batch, d_model, seq_len)
        pooled = self.pool(output).squeeze(-1)  # (batch, d_model)

        pooled = self.layernorm(pooled)
        out = self.fc(pooled)  # (batch, 1)
        return out.squeeze(-1)

# 損失函數 訓練與驗證函數
def train_epoch(model, optimizer, loss_fn, dataloader):
    model.train()
    total_loss = 0
    for xb, yb in dataloader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb.squeeze())
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(dataloader.dataset)

def val_epoch(model, loss_fn, dataloader):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for xb, yb in dataloader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = loss_fn(pred, yb.squeeze())
            total_loss += loss.item() * xb.size(0)
    return total_loss / len(dataloader.dataset)

# 評估指標函數
def evaluate_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    return mse, mae, rmse

# 訓練超參數
epochs = 30

# 建立與訓練 LSTM
input_dim = len(feature_cols)
lstm_model = LSTMModel(input_dim).to(device)
optimizer_lstm = optim.Adam(lstm_model.parameters(), lr=1e-4)
loss_fn = nn.MSELoss()

train_losses_lstm = []
val_losses_lstm = []

for epoch in range(epochs):
    train_loss = train_epoch(lstm_model, optimizer_lstm, loss_fn, train_loader)
    val_loss = val_epoch(lstm_model, loss_fn, val_loader)
    train_losses_lstm.append(train_loss)
    val_losses_lstm.append(val_loss)
    print(f"LSTM Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}")

# 調整 Transformer optimizer 和 scheduler
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

transformer_model = TransformerEncoderModel(input_dim).to(device)
optimizer_transformer = optim.AdamW(transformer_model.parameters(), lr=3e-4, weight_decay=1e-4)

# Warmup scheduler for first 10% epochs
class WarmupScheduler:
    def __init__(self, optimizer, warmup_steps, base_lr):
        self.optimizer = optimizer
        self.warmup_steps = warmup_steps
        self.base_lr = base_lr
        self.step_num = 0

    def step(self):
        self.step_num += 1
        if self.step_num <= self.warmup_steps:
            lr = self.base_lr * self.step_num / self.warmup_steps
            for param_group in self.optimizer.param_groups:
                param_group['lr'] = lr

warmup_steps = int(epochs * 0.1)
warmup_scheduler = WarmupScheduler(optimizer_transformer, warmup_steps, 3e-4)
cosine_scheduler = CosineAnnealingWarmRestarts(optimizer_transformer, T_0=epochs - warmup_steps)

# Transformer 訓練函數更新，包含 scheduler step
def train_epoch_transformer(model, optimizer, loss_fn, dataloader, warmup_scheduler, cosine_scheduler, epoch):
    model.train()
    total_loss = 0
    for xb, yb in dataloader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb.squeeze())
        loss.backward()
        optimizer.step()
        # scheduler step
        if epoch < warmup_steps:
            warmup_scheduler.step()
        else:
            cosine_scheduler.step()
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(dataloader.dataset)

val_losses_transformer = []
train_losses_transformer = []

for epoch in range(epochs):
    train_loss = train_epoch_transformer(transformer_model, optimizer_transformer, loss_fn, train_loader,
                                        warmup_scheduler, cosine_scheduler, epoch)
    val_loss = val_epoch(transformer_model, loss_fn, val_loader)
    train_losses_transformer.append(train_loss)
    val_losses_transformer.append(val_loss)
    print(f"Transformer Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}")

# 評估並反標準化預測值（以方便轉回價格尺度）
def predict(model, dataloader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for xb, yb in dataloader:
            xb = xb.to(device)
            pred = model(xb)
            y_true.append(yb.numpy())
            y_pred.append(pred.cpu().numpy())
    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    return y_true, y_pred

# LSTM 預測與評估
y_true_train, y_pred_train = predict(lstm_model, train_loader)
y_true_val, y_pred_val = predict(lstm_model, val_loader)
y_true_test, y_pred_test = predict(lstm_model, test_loader)

# 反標準化log return 預測與實際值
def inverse_transform_logret(logret_scaled):
    logret_scaled = logret_scaled.reshape(-1, 1)
    return scaler_y.inverse_transform(logret_scaled).flatten()

y_true_test_orig = inverse_transform_logret(y_true_test)
y_pred_test_orig = inverse_transform_logret(y_pred_test)

# 將log return 累積還原為價格
def logret_to_price(logret, start_price):
    price = [start_price]
    for lr in logret:
        price.append(price[-1] * np.exp(lr))
    return np.array(price[1:])  # 去除起始價

start_price = test_df["Close"].values[0]

price_true = logret_to_price(y_true_test_orig, start_price)
price_pred = logret_to_price(y_pred_test_orig, start_price)

# 標準化空間與價格尺度指標計算函數
def calc_all_metrics(y_true_scaled, y_pred_scaled, y_true_orig, y_pred_orig):
    mse_std, mae_std, rmse_std = evaluate_metrics(y_true_scaled, y_pred_scaled)
    mse_orig, mae_orig, rmse_orig = evaluate_metrics(y_true_orig, y_pred_orig)
    return {"MSE Std": mse_std, "MAE Std": mae_std, "RMSE Std": rmse_std,
            "MSE Orig": mse_orig, "MAE Orig": mae_orig, "RMSE Orig": rmse_orig}

lstm_metrics = calc_all_metrics(y_true_test, y_pred_test, price_true, price_pred)

# Transformer 預測與評估
y_true_test_tf, y_pred_test_tf = predict(transformer_model, test_loader)
y_true_test_tf_orig = inverse_transform_logret(y_true_test_tf)
y_pred_test_tf_orig = inverse_transform_logret(y_pred_test_tf)
price_true_tf = logret_to_price(y_true_test_tf_orig, start_price)
price_pred_tf = logret_to_price(y_pred_test_tf_orig, start_price)

transformer_metrics = calc_all_metrics(y_true_test_tf, y_pred_test_tf, price_true_tf, price_pred_tf)

# 繪製預測對比圖 LSTM 與 Transformer
plt.figure(figsize=(14, 6))
plt.plot(price_true, label="True Price (Test)")
plt.plot(price_pred, label="LSTM Predicted Price")
plt.plot(price_pred_tf, label="Transformer Predicted Price")
plt.title("True vs Predicted Price (Test Set)")
plt.legend()
plt.show()

# 繪製學習曲線
plt.figure(figsize=(12, 5))
plt.plot(range(1, epochs+1), np.sqrt(train_losses_lstm), label="LSTM Train RMSE")
plt.plot(range(1, epochs+1), np.sqrt(val_losses_lstm), label="LSTM Val RMSE")
plt.plot(range(1, epochs+1), np.sqrt(train_losses_transformer), label="Transformer Train RMSE")
plt.plot(range(1, epochs+1), np.sqrt(val_losses_transformer), label="Transformer Val RMSE")
plt.xlabel("Epoch")
plt.ylabel("RMSE")
plt.title("Training and Validation RMSE")
plt.legend()
plt.show()

# Transformer Attention Heatmap (示範取第一個batch的注意力權重)
def plot_attention_heatmap(model):
    model.eval()
    with torch.no_grad():
        for xb, _ in test_loader:
            xb = xb.to(device)
            model(xb)
            attn_weights = model.attn_weights
            break

    if attn_weights is None:
        print("未實作取得 Transformer 注意力權重，請補充相關方法以取得 heatmap")
        return

    attn_weights = attn_weights.cpu().numpy()
    plt.figure(figsize=(8, 6))
    plt.imshow(attn_weights[0], cmap="viridis", aspect="auto")
    plt.colorbar()
    plt.title("Transformer Attention Heatmap (Sample)")
    plt.xlabel("Key")
    plt.ylabel("Query")
    plt.show()

# 目前沒有直接回傳注意力權重，需要額外hook或修改transformer架構來獲取，這部分可進一步實作，文中提及熱力圖。

# 顯示評估結果
print("LSTM 評估指標 (Test Set):")
for k, v in lstm_metrics.items():
    print(f"{k}: {v:.6f}")

print("\nTransformer 評估指標 (Test Set):")
for k, v in transformer_metrics.items():
    print(f"{k}: {v:.6f}")


In [ ]:
# 安装必要的库
!pip install yfinance ta

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from ta.trend import SMAIndicator, MACD
from ta.momentum import RSIIndicator
from ta.volatility import BollingerBands
import warnings
warnings.filterwarnings('ignore')

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

# 设备配置
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 1. 数据获取和预处理
def download_data(ticker='2330.TW', start_date='2020-01-01', end_date='2024-12-31'):
    """下载股票数据"""
    stock = yf.download(ticker, start=start_date, end=end_date)
    return stock

def calculate_technical_indicators(df):
    """计算技术指标"""
    # 基础价格数据
    df['SMA_5'] = SMAIndicator(close=df['Close'], window=5).sma_indicator()
    df['SMA_20'] = SMAIndicator(close=df['Close'], window=20).sma_indicator()

    # RSI
    df['RSI_14'] = RSIIndicator(close=df['Close'], window=14).rsi()

    # MACD
    macd = MACD(close=df['Close'])
    df['MACD'] = macd.macd()
    df['MACD_SIGNAL'] = macd.macd_signal()
    df['MACD_HIST'] = macd.macd_diff()

    return df

def prepare_target(df):
    """准备目标变量 - log return"""
    df['logret'] = np.log(df['Close'].shift(-1)) - np.log(df['Close'])
    return df

def create_sequences(data, targets, lookback=30):
    """创建时间序列数据"""
    X, y = [], []
    for i in range(len(data) - lookback):
        X.append(data[i:(i + lookback)])
        y.append(targets[i + lookback])
    return np.array(X), np.array(y)

# 2. 数据集类
class StockDataset(Dataset):
    def __init__(self, features, targets):
        self.features = torch.FloatTensor(features)
        self.targets = torch.FloatTensor(targets)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]

# 3. 模型定义
# LSTM 模型
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, dropout=0.1):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True, dropout=dropout)
        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.linear = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        lstm_out, (h_n, c_n) = self.lstm(x)
        last_output = lstm_out[:, -1, :]
        normalized = self.layer_norm(last_output)
        output = self.linear(normalized)
        return output.squeeze(-1)

# Transformer 位置编码
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)

        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:x.size(0), :]

# Transformer 模型
class TransformerModel(nn.Module):
    def __init__(self, input_dim, d_model=128, nhead=8, num_layers=2, dim_feedforward=256, dropout=0.1):
        super(TransformerModel, self).__init__()
        self.d_model = d_model

        # 输入投影
        self.input_projection = nn.Linear(input_dim, d_model)
        self.pos_encoding = PositionalEncoding(d_model)

        # Transformer编码器
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 输出层
        self.layer_norm = nn.LayerNorm(d_model)
        self.linear = nn.Linear(d_model, 1)

    def forward(self, x, return_attention=False):
        # 输入投影和位置编码
        x = self.input_projection(x) * np.sqrt(self.d_model)
        x = self.pos_encoding(x)

        # Transformer编码器
        if return_attention:
            # 注册hook来获取注意力权重
            attention_weights = []
            def get_attention(module, input, output):
                attention_weights.append(output[1])  # 输出是 (output, attention_weights)

            handles = []
            for layer in self.transformer_encoder.layers:
                handles.append(layer.self_attn.register_forward_hook(get_attention))

            transformer_output = self.transformer_encoder(x)

            # 移除hooks
            for handle in handles:
                handle.remove()
        else:
            transformer_output = self.transformer_encoder(x)
            attention_weights = None

        # Mean pooling和输出
        pooled = transformer_output.mean(dim=1)
        normalized = self.layer_norm(pooled)
        output = self.linear(normalized)

        if return_attention:
            return output.squeeze(-1), attention_weights
        else:
            return output.squeeze(-1)

# 4. 训练函数
def train_model(model, train_loader, val_loader, optimizer, criterion, epochs=30, model_type='lstm', scheduler=None):
    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        # 训练阶段
        model.train()
        train_loss = 0
        for batch_features, batch_targets in train_loader:
            batch_features, batch_targets = batch_features.to(device), batch_targets.to(device)

            optimizer.zero_grad()
            outputs = model(batch_features)
            loss = criterion(outputs, batch_targets)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        # 验证阶段
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch_features, batch_targets in val_loader:
                batch_features, batch_targets = batch_features.to(device), batch_targets.to(device)
                outputs = model(batch_features)
                loss = criterion(outputs, batch_targets)
                val_loss += loss.item()

        # 学习率调度
        if scheduler:
            if model_type == 'transformer':
                scheduler.step()
            elif epoch >= epochs * 0.1:  # LSTM在10% epochs后开始调度
                scheduler.step()

        train_loss /= len(train_loader)
        val_loss /= len(val_loader)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        if (epoch + 1) % 5 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}')

    return train_losses, val_losses

# 5. 评估函数
def evaluate_model(model, test_loader, scaler_target, original_close_prices, test_indices, lookback):
    model.eval()
    predictions = []
    actuals = []

    with torch.no_grad():
        for batch_features, batch_targets in test_loader:
            batch_features = batch_features.to(device)
            outputs = model(batch_features)
            predictions.extend(outputs.cpu().numpy())
            actuals.extend(batch_targets.numpy())

    # 转换为numpy数组
    predictions = np.array(predictions)
    actuals = np.array(actuals)

    # 反标准化目标变量
    predictions_original = scaler_target.inverse_transform(predictions.reshape(-1, 1)).flatten()
    actuals_original = scaler_target.inverse_transform(actuals.reshape(-1, 1)).flatten()

    # 转换回原始价格
    pred_prices = []
    true_prices = []

    for i in range(len(predictions_original)):
        pred_logret = predictions_original[i]
        true_logret = actuals_original[i]

        # 使用前一天的收盘价计算预测的收盘价
        prev_close = original_close_prices[test_indices[i] - 1]  # lookback序列的最后一天
        pred_close = prev_close * np.exp(pred_logret)
        true_close = prev_close * np.exp(true_logret)

        pred_prices.append(pred_close)
        true_prices.append(true_close)

    return (predictions, actuals), (np.array(pred_prices), np.array(true_prices))

# 6. 指标计算
def calculate_metrics(predictions, actuals, scale_name=""):
    mse = mean_squared_error(actuals, predictions)
    mae = mean_absolute_error(actuals, predictions)
    rmse = np.sqrt(mse)

    print(f"{scale_name} Metrics:")
    print(f"MSE: {mse:.6f}")
    print(f"MAE: {mae:.6f}")
    print(f"RMSE: {rmse:.6f}")
    print()

    return mse, mae, rmse

# 7. 可视化函数
def plot_predictions(true_values, pred_values, model_name, dates=None):
    plt.figure(figsize=(12, 6))

    if dates is not None:
        plt.plot(dates, true_values, label='True Prices', alpha=0.7)
        plt.plot(dates, pred_values, label='Predicted Prices', alpha=0.7)
    else:
        plt.plot(true_values, label='True Prices', alpha=0.7)
        plt.plot(pred_values, label='Predicted Prices', alpha=0.7)

    plt.title(f'{model_name}: True vs Predicted Prices')
    plt.xlabel('Time')
    plt.ylabel('Price')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

def plot_attention_heatmap(attention_weights, layer_idx=0, head_idx=0):
    """绘制注意力热图"""
    if len(attention_weights) > 0:
        # 获取指定层和头的注意力权重
        attention = attention_weights[layer_idx][head_idx].cpu().numpy()

        plt.figure(figsize=(10, 8))
        sns.heatmap(attention, cmap='viridis', cbar=True)
        plt.title(f'Attention Heatmap - Layer {layer_idx}, Head {head_idx}')
        plt.xlabel('Key Position')
        plt.ylabel('Query Position')
        plt.show()

def plot_learning_curves(train_losses, val_losses, model_name):
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.title(f'{model_name} Learning Curve')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# 主函数
def main():
    # 下载和处理数据
    print("Downloading data...")
    df = download_data('2330.TW', '2020-01-01', '2024-12-31')

    print("Calculating technical indicators...")
    df = calculate_technical_indicators(df)

    print("Preparing target variable...")
    df = prepare_target(df)

    # 选择特征
    feature_columns = ['Open', 'High', 'Low', 'Close', 'Volume',
                      'SMA_5', 'SMA_20', 'RSI_14', 'MACD', 'MACD_SIGNAL', 'MACD_HIST']

    # 移除NaN值
    df_clean = df[feature_columns + ['logret']].dropna()

    # 时间序列分割 (8:1:1)
    total_len = len(df_clean)
    train_size = int(total_len * 0.8)
    val_size = int(total_len * 0.1)

    train_data = df_clean.iloc[:train_size]
    val_data = df_clean.iloc[train_size:train_size + val_size]
    test_data = df_clean.iloc[train_size + val_size:]

    print(f"Train size: {len(train_data)}, Val size: {len(val_data)}, Test size: {len(test_data)}")

    # 标准化特征和目标
    scaler_features = StandardScaler()
    scaler_target = StandardScaler()

    # 只在训练集上拟合scaler
    train_features_scaled = scaler_features.fit_transform(train_data[feature_columns])
    train_target_scaled = scaler_target.fit_transform(train_data[['logret']])

    val_features_scaled = scaler_features.transform(val_data[feature_columns])
    val_target_scaled = scaler_target.transform(val_data[['logret']])

    test_features_scaled = scaler_features.transform(test_data[feature_columns])
    test_target_scaled = scaler_target.transform(test_data[['logret']])

    # 创建序列数据
    lookback = 30
    X_train, y_train = create_sequences(train_features_scaled, train_target_scaled, lookback)
    X_val, y_val = create_sequences(val_features_scaled, val_target_scaled, lookback)
    X_test, y_test = create_sequences(test_features_scaled, test_target_scaled, lookback)

    print(f"Training sequences: {X_train.shape}")
    print(f"Validation sequences: {X_val.shape}")
    print(f"Test sequences: {X_test.shape}")

    # 创建数据加载器
    batch_size = 64
    train_dataset = StockDataset(X_train, y_train)
    val_dataset = StockDataset(X_val, y_val)
    test_dataset = StockDataset(X_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)  # 时间序列不打乱
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # 模型参数
    input_dim = len(feature_columns)

    # 训练LSTM模型
    print("\n" + "="*50)
    print("Training LSTM Model")
    print("="*50)

    lstm_model = LSTMModel(input_dim=input_dim).to(device)
    lstm_optimizer = optim.Adam(lstm_model.parameters(), lr=1e-4)
    lstm_criterion = nn.MSELoss()
    lstm_scheduler = optim.lr_scheduler.StepLR(lstm_optimizer, step_size=10, gamma=0.5)

    lstm_train_losses, lstm_val_losses = train_model(
        lstm_model, train_loader, val_loader, lstm_optimizer, lstm_criterion,
        epochs=30, model_type='lstm', scheduler=lstm_scheduler
    )

    # 训练Transformer模型
    print("\n" + "="*50)
    print("Training Transformer Model")
    print("="*50)

    transformer_model = TransformerModel(input_dim=input_dim).to(device)
    transformer_optimizer = optim.AdamW(transformer_model.parameters(), lr=3e-4, weight_decay=1e-4)
    transformer_criterion = nn.MSELoss()

    # CosineAnnealingWarmRestarts调度器
    transformer_scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        transformer_optimizer, T_0=3, T_mult=2
    )

    transformer_train_losses, transformer_val_losses = train_model(
        transformer_model, train_loader, val_loader, transformer_optimizer, transformer_criterion,
        epochs=30, model_type='transformer', scheduler=transformer_scheduler
    )

    # 评估模型
    print("\n" + "="*50)
    print("Model Evaluation")
    print("="*50)

    # 获取测试集索引用于价格转换
    test_indices = list(range(lookback, len(test_data)))

    # 评估LSTM
    print("LSTM Evaluation:")
    (lstm_pred_scaled, lstm_actual_scaled), (lstm_pred_prices, lstm_actual_prices) = evaluate_model(
        lstm_model, test_loader, scaler_target, df_clean['Close'].values,
        test_indices, lookback
    )

    # 评估Transformer（获取注意力权重）
    print("Transformer Evaluation:")
    transformer_model.eval()
    transformer_predictions = []
    transformer_actuals = []
    attention_weights_list = []

    with torch.no_grad():
        for batch_features, batch_targets in test_loader:
            batch_features = batch_features.to(device)
            outputs, attention_weights = transformer_model(batch_features, return_attention=True)
            transformer_predictions.extend(outputs.cpu().numpy())
            transformer_actuals.extend(batch_targets.numpy())
            attention_weights_list.append(attention_weights)

    transformer_predictions = np.array(transformer_predictions)
    transformer_actuals = np.array(transformer_actuals)

    # 反标准化
    transformer_pred_original = scaler_target.inverse_transform(transformer_predictions.reshape(-1, 1)).flatten()
    transformer_actual_original = scaler_target.inverse_transform(transformer_actuals.reshape(-1, 1)).flatten()

    # 转换回价格
    transformer_pred_prices = []
    transformer_actual_prices = []

    for i in range(len(transformer_pred_original)):
        pred_logret = transformer_pred_original[i]
        true_logret = transformer_actual_original[i]

        prev_close = df_clean['Close'].iloc[test_indices[i] - 1]
        pred_close = prev_close * np.exp(pred_logret)
        true_close = prev_close * np.exp(true_logret)

        transformer_pred_prices.append(pred_close)
        transformer_actual_prices.append(true_close)

    transformer_pred_prices = np.array(transformer_pred_prices)
    transformer_actual_prices = np.array(transformer_actual_prices)

    # 计算指标
    print("\nLSTM Metrics:")
    print("Standardized Scale:")
    lstm_mse_scaled, lstm_mae_scaled, lstm_rmse_scaled = calculate_metrics(
        lstm_pred_scaled, lstm_actual_scaled, "Standardized"
    )

    print("Original Price Scale:")
    lstm_mse_price, lstm_mae_price, lstm_rmse_price = calculate_metrics(
        lstm_pred_prices, lstm_actual_prices, "Price"
    )

    print("\nTransformer Metrics:")
    print("Standardized Scale:")
    transformer_mse_scaled, transformer_mae_scaled, transformer_rmse_scaled = calculate_metrics(
        transformer_predictions, transformer_actuals, "Standardized"
    )

    print("Original Price Scale:")
    transformer_mse_price, transformer_mae_price, transformer_rmse_price = calculate_metrics(
        transformer_pred_prices, transformer_actual_prices, "Price"
    )

    # 可视化结果
    print("\n" + "="*50)
    print("Visualizations")
    print("="*50)

    # LSTM预测图
    plot_predictions(lstm_actual_prices, lstm_pred_prices, "LSTM")

    # Transformer预测图
    plot_predictions(transformer_actual_prices, transformer_pred_prices, "Transformer")

    # 注意力热图
    if len(attention_weights_list) > 0:
        # 使用第一个batch的第一个样本
        sample_attention = attention_weights_list[0]
        plot_attention_heatmap(sample_attention, layer_idx=0, head_idx=0)

    # 学习曲线
    plot_learning_curves(lstm_train_losses, lstm_val_losses, "LSTM")
    plot_learning_curves(transformer_train_losses, transformer_val_losses, "Transformer")

    # 技术指标分析
    print("\n" + "="*50)
    print("Technical Indicators Analysis")
    print("="*50)

    # 分析不同技术指标对预测的贡献
    tech_indicators = {
        'Trend': ['SMA_5', 'SMA_20'],
        'Momentum': ['RSI_14'],
        'Oscillator': ['MACD', 'MACD_SIGNAL', 'MACD_HIST'],
        'Volume': ['Volume']
    }

    for category, indicators in tech_indicators.items():
        print(f"\n{category} Indicators: {indicators}")
        # 这里可以添加更详细的分析，比如特征重要性分析

    # 保存结果
    results = {
        'LSTM': {
            'scaled': {'MSE': lstm_mse_scaled, 'MAE': lstm_mae_scaled, 'RMSE': lstm_rmse_scaled},
            'price': {'MSE': lstm_mse_price, 'MAE': lstm_mae_price, 'RMSE': lstm_rmse_price}
        },
        'Transformer': {
            'scaled': {'MSE': transformer_mse_scaled, 'MAE': transformer_mae_scaled, 'RMSE': transformer_rmse_scaled},
            'price': {'MSE': transformer_mse_price, 'MAE': transformer_mae_price, 'RMSE': transformer_rmse_price}
        }
    }

    return results, df_clean

# 运行主函数
if __name__ == "__main__":
    results, processed_data = main()

    # 打印最终结果摘要
    print("\n" + "="*50)
    print("FINAL RESULTS SUMMARY")
    print("="*50)

    print("\nLSTM Performance:")
    print(f"Standardized - MSE: {results['LSTM']['scaled']['MSE']:.6f}, "
          f"MAE: {results['LSTM']['scaled']['MAE']:.6f}, "
          f"RMSE: {results['LSTM']['scaled']['RMSE']:.6f}")
    print(f"Price Scale - MSE: {results['LSTM']['price']['MSE']:.6f}, "
          f"MAE: {results['LSTM']['price']['MAE']:.6f}, "
          f"RMSE: {results['LSTM']['price']['RMSE']:.6f}")

    print("\nTransformer Performance:")
    print(f"Standardized - MSE: {results['Transformer']['scaled']['MSE']:.6f}, "
          f"MAE: {results['Transformer']['scaled']['MAE']:.6f}, "
          f"RMSE: {results['Transformer']['scaled']['RMSE']:.6f}")
    print(f"Price Scale - MSE: {results['Transformer']['price']['MSE']:.6f}, "
          f"MAE: {results['Transformer']['price']['MAE']:.6f}, "
          f"RMSE: {results['Transformer']['price']['RMSE']:.6f}")

In [ ]:
# ============================================================================
# HW7: 利用 LSTM 與 Transformer 預測股價時間序列
# ============================================================================

# 安裝必要套件
!pip install yfinance ta -q

# 導入套件
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

# 設定隨機種子
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# ============================================================================
# 1. 資料下載與特徵工程
# ============================================================================

def download_stock_data(ticker='2330.TW', start='2020-01-01', end='2024-12-31'):
    """下載股價資料"""
    print(f"下載 {ticker} 資料...")
    df = yf.download(ticker, start=start, end=end)
    df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
    df = df.dropna()
    return df

def add_technical_indicators(df):
    """新增技術指標（四類）"""
    df = df.copy()

    # 趨勢型指標
    df['SMA_5'] = df['Close'].rolling(window=5).mean()
    df['SMA_20'] = df['Close'].rolling(window=20).mean()
    df['EMA_12'] = df['Close'].ewm(span=12, adjust=False).mean()
    df['EMA_26'] = df['Close'].ewm(span=26, adjust=False).mean()

    # MACD (趨勢型)
    df['MACD'] = df['EMA_12'] - df['EMA_26']
    df['MACD_SIGNAL'] = df['MACD'].ewm(span=9, adjust=False).mean()
    df['MACD_HIST'] = df['MACD'] - df['MACD_SIGNAL']

    # 震盪型指標
    # RSI
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['RSI_14'] = 100 - (100 / (1 + rs))

    # Stochastic Oscillator
    low_14 = df['Low'].rolling(window=14).min()
    high_14 = df['High'].rolling(window=14).max()
    df['STOCH_K'] = 100 * (df['Close'] - low_14) / (high_14 - low_14)
    df['STOCH_D'] = df['STOCH_K'].rolling(window=3).mean()

    # 波動型指標
    # Bollinger Bands
    df['BB_MIDDLE'] = df['Close'].rolling(window=20).mean()
    bb_std = df['Close'].rolling(window=20).std()
    df['BB_UPPER'] = df['BB_MIDDLE'] + 2 * bb_std
    df['BB_LOWER'] = df['BB_MIDDLE'] - 2 * bb_std
    df['BB_WIDTH'] = (df['BB_UPPER'] - df['BB_LOWER']) / df['BB_MIDDLE']

    # ATR (Average True Range)
    high_low = df['High'] - df['Low']
    high_close = np.abs(df['High'] - df['Close'].shift())
    low_close = np.abs(df['Low'] - df['Close'].shift())
    ranges = pd.concat([high_low, high_close, low_close], axis=1)
    true_range = np.max(ranges, axis=1)
    df['ATR_14'] = true_range.rolling(14).mean()

    # 量能型指標
    # OBV (On-Balance Volume)
    df['OBV'] = (np.sign(df['Close'].diff()) * df['Volume']).fillna(0).cumsum()

    # Volume SMA
    df['VOLUME_SMA_20'] = df['Volume'].rolling(window=20).mean()
    df['VOLUME_RATIO'] = df['Volume'] / df['VOLUME_SMA_20']

    # 目標變數：log return
    df['logret'] = np.log(df['Close']) - np.log(df['Close'].shift(1))

    # 移除 NaN
    df = df.dropna()

    return df

# 下載資料
df = download_stock_data('2330.TW', '2020-01-01', '2024-12-31')
df = add_technical_indicators(df)

print(f"資料形狀: {df.shape}")
print(f"資料期間: {df.index[0]} 到 {df.index[-1]}")
print(f"\n欄位: {df.columns.tolist()}")

# ============================================================================
# 2. 資料準備與時間切分
# ============================================================================

def prepare_data(df, lookback=30):
    """準備時間序列資料"""
    # 特徵欄位（排除目標變數）
    feature_cols = [col for col in df.columns if col not in ['logret']]

    # 目標變數（未來一天的 log return）
    df['target'] = df['logret'].shift(-1)
    df = df.dropna()

    # 時間序切分 8:1:1
    n = len(df)
    train_size = int(n * 0.8)
    val_size = int(n * 0.1)

    train_df = df.iloc[:train_size]
    val_df = df.iloc[train_size:train_size+val_size]
    test_df = df.iloc[train_size+val_size:]

    print(f"訓練集: {len(train_df)} 樣本")
    print(f"驗證集: {len(val_df)} 樣本")
    print(f"測試集: {len(test_df)} 樣本")

    # 標準化（只用訓練集 fit）
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train = scaler_X.fit_transform(train_df[feature_cols])
    y_train = scaler_y.fit_transform(train_df[['target']])

    X_val = scaler_X.transform(val_df[feature_cols])
    y_val = scaler_y.transform(val_df[['target']])

    X_test = scaler_X.transform(test_df[feature_cols])
    y_test = scaler_y.transform(test_df[['target']])

    # 建立序列資料
    def create_sequences(X, y, lookback):
        Xs, ys = [], []
        for i in range(len(X) - lookback):
            Xs.append(X[i:i+lookback])
            ys.append(y[i+lookback])
        return np.array(Xs), np.array(ys)

    X_train_seq, y_train_seq = create_sequences(X_train, y_train, lookback)
    X_val_seq, y_val_seq = create_sequences(X_val, y_val, lookback)
    X_test_seq, y_test_seq = create_sequences(X_test, y_test, lookback)

    return {
        'X_train': X_train_seq, 'y_train': y_train_seq,
        'X_val': X_val_seq, 'y_val': y_val_seq,
        'X_test': X_test_seq, 'y_test': y_test_seq,
        'scaler_X': scaler_X, 'scaler_y': scaler_y,
        'feature_cols': feature_cols,
        'train_df': train_df, 'val_df': val_df, 'test_df': test_df
    }

lookback = 30
data_dict = prepare_data(df, lookback=lookback)

print(f"\n訓練序列形狀: {data_dict['X_train'].shape}")
print(f"驗證序列形狀: {data_dict['X_val'].shape}")
print(f"測試序列形狀: {data_dict['X_test'].shape}")

# ============================================================================
# 3. Dataset 與 DataLoader
# ============================================================================

class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = StockDataset(data_dict['X_train'], data_dict['y_train'])
val_dataset = StockDataset(data_dict['X_val'], data_dict['y_val'])
test_dataset = StockDataset(data_dict['X_test'], data_dict['y_test'])

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# ============================================================================
# 4. 模型定義
# ============================================================================

# LSTM Baseline
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, dropout=0.1):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True, dropout=dropout)
        self.layer_norm = nn.LayerNorm(hidden_size)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        lstm_out, (h_n, c_n) = self.lstm(x)
        # 取最後一個時間步
        last_hidden = lstm_out[:, -1, :]
        normalized = self.layer_norm(last_hidden)
        output = self.fc(normalized)
        return output

# Transformer (Encoder-only)
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        return x + self.pe[:, :x.size(1), :]

class TransformerModel(nn.Module):
    def __init__(self, input_size, d_model=128, nhead=4, num_layers=2,
                 dim_feedforward=256, dropout=0.1):
        super(TransformerModel, self).__init__()
        self.input_projection = nn.Linear(input_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, 1)
        self.attention_weights = None

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        x = self.input_projection(x)
        x = self.pos_encoder(x)

        # 儲存注意力權重
        encoded = self.transformer_encoder(x)

        # Mean pooling
        pooled = encoded.mean(dim=1)
        output = self.fc(pooled)

        return output

    def get_attention_weights(self, x):
        """取得注意力權重用於視覺化"""
        x = self.input_projection(x)
        x = self.pos_encoder(x)

        # 手動通過每一層並收集注意力權重
        attentions = []
        for layer in self.transformer_encoder.layers:
            # 使用 self-attention
            attn_output, attn_weights = layer.self_attn(x, x, x, average_attn_weights=True)
            attentions.append(attn_weights.detach().cpu())
            x = layer(x)

        return attentions

# ============================================================================
# 5. 訓練函數
# ============================================================================

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(X_batch)

    return total_loss / len(loader.dataset)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    predictions = []
    actuals = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            total_loss += loss.item() * len(X_batch)
            predictions.append(outputs.cpu().numpy())
            actuals.append(y_batch.cpu().numpy())

    predictions = np.concatenate(predictions, axis=0)
    actuals = np.concatenate(actuals, axis=0)

    return total_loss / len(loader.dataset), predictions, actuals

def train_model(model, train_loader, val_loader, criterion, optimizer,
                scheduler, epochs, device, model_name):
    """訓練模型"""
    train_losses = []
    val_losses = []
    val_rmses = []

    best_val_loss = float('inf')
    best_model_state = None

    print(f"\n開始訓練 {model_name}...")
    for epoch in range(epochs):
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_preds, val_actuals = evaluate(model, val_loader, criterion, device)

        val_rmse = np.sqrt(mean_squared_error(val_actuals, val_preds))

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        val_rmses.append(val_rmse)

        if scheduler is not None:
            scheduler.step()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()

        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.6f}, "
                  f"Val Loss: {val_loss:.6f}, Val RMSE: {val_rmse:.6f}")

    # 載入最佳模型
    model.load_state_dict(best_model_state)

    return model, train_losses, val_losses, val_rmses

# ============================================================================
# 6. 訓練 LSTM
# ============================================================================

input_size = data_dict['X_train'].shape[2]
epochs = 30

# LSTM
lstm_model = LSTMModel(input_size=input_size, hidden_size=128, dropout=0.1).to(device)
lstm_criterion = nn.MSELoss()
lstm_optimizer = optim.Adam(lstm_model.parameters(), lr=1e-4)

lstm_model, lstm_train_losses, lstm_val_losses, lstm_val_rmses = train_model(
    lstm_model, train_loader, val_loader, lstm_criterion, lstm_optimizer,
    scheduler=None, epochs=epochs, device=device, model_name="LSTM"
)

# ============================================================================
# 7. 訓練 Transformer
# ============================================================================

# Transformer
transformer_model = TransformerModel(
    input_size=input_size,
    d_model=128,
    nhead=4,
    num_layers=2,
    dim_feedforward=256,
    dropout=0.1
).to(device)

transformer_criterion = nn.MSELoss()
transformer_optimizer = optim.AdamW(transformer_model.parameters(), lr=3e-4, weight_decay=1e-4)

# Warmup + CosineAnnealing
warmup_epochs = int(0.1 * epochs)
transformer_scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    transformer_optimizer, T_0=10, T_mult=2
)

transformer_model, transformer_train_losses, transformer_val_losses, transformer_val_rmses = train_model(
    transformer_model, train_loader, val_loader, transformer_criterion,
    transformer_optimizer, transformer_scheduler, epochs=epochs,
    device=device, model_name="Transformer"
)

# ============================================================================
# 8. 測試集評估
# ============================================================================

def compute_metrics(y_true, y_pred, scaler_y, model_name):
    """計算標準化空間與原始尺度的指標"""
    # 標準化空間
    mse_norm = mean_squared_error(y_true, y_pred)
    mae_norm = mean_absolute_error(y_true, y_pred)
    rmse_norm = np.sqrt(mse_norm)

    # 反標準化到原始尺度
    y_true_orig = scaler_y.inverse_transform(y_true)
    y_pred_orig = scaler_y.inverse_transform(y_pred)

    mse_orig = mean_squared_error(y_true_orig, y_pred_orig)
    mae_orig = mean_absolute_error(y_true_orig, y_pred_orig)
    rmse_orig = np.sqrt(mse_orig)

    print(f"\n{model_name} 測試集結果:")
    print("=" * 60)
    print("標準化空間:")
    print(f"  MSE:  {mse_norm:.6f}")
    print(f"  MAE:  {mae_norm:.6f}")
    print(f"  RMSE: {rmse_norm:.6f}")
    print("\n原始價格尺度 (log return):")
    print(f"  MSE:  {mse_orig:.6f}")
    print(f"  MAE:  {mae_orig:.6f}")
    print(f"  RMSE: {rmse_orig:.6f}")

    return {
        'mse_norm': mse_norm, 'mae_norm': mae_norm, 'rmse_norm': rmse_norm,
        'mse_orig': mse_orig, 'mae_orig': mae_orig, 'rmse_orig': rmse_orig,
        'y_true_orig': y_true_orig, 'y_pred_orig': y_pred_orig
    }

# LSTM 測試
_, lstm_test_preds, lstm_test_actuals = evaluate(
    lstm_model, test_loader, lstm_criterion, device
)
lstm_metrics = compute_metrics(
    lstm_test_actuals, lstm_test_preds,
    data_dict['scaler_y'], "LSTM"
)

# Transformer 測試
_, transformer_test_preds, transformer_test_actuals = evaluate(
    transformer_model, test_loader, transformer_criterion, device
)
transformer_metrics = compute_metrics(
    transformer_test_actuals, transformer_test_preds,
    data_dict['scaler_y'], "Transformer"
)

# ============================================================================
# 9. 視覺化
# ============================================================================

# 9.1 學習曲線
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# LSTM
axes[0].plot(lstm_val_rmses, label='LSTM Val RMSE', color='blue')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('RMSE (Normalized)')
axes[0].set_title('LSTM Validation Learning Curve')
axes[0].legend()
axes[0].grid(True)

# Transformer
axes[1].plot(transformer_val_rmses, label='Transformer Val RMSE', color='red')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('RMSE (Normalized)')
axes[1].set_title('Transformer Validation Learning Curve')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('learning_curves.png', dpi=300, bbox_inches='tight')
plt.show()

# 9.2 預測結果對比（原始尺度）
def convert_logret_to_price(logret_series, initial_price):
    """將 log return 轉換回價格"""
    prices = [initial_price]
    for lr in logret_series:
        prices.append(prices[-1] * np.exp(lr))
    return np.array(prices[1:])

# 取得測試集的初始價格
test_start_idx = len(data_dict['train_df']) + len(data_dict['val_df']) + lookback
test_initial_price = df['Close'].iloc[test_start_idx]

# 轉換為價格
lstm_true_prices = convert_logret_to_price(
    lstm_metrics['y_true_orig'].flatten(), test_initial_price
)
lstm_pred_prices = convert_logret_to_price(
    lstm_metrics['y_pred_orig'].flatten(), test_initial_price
)

transformer_true_prices = convert_logret_to_price(
    transformer_metrics['y_true_orig'].flatten(), test_initial_price
)
transformer_pred_prices = convert_logret_to_price(
    transformer_metrics['y_pred_orig'].flatten(), test_initial_price
)

fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# LSTM
axes[0].plot(lstm_true_prices, label='True Price', color='black', linewidth=2)
axes[0].plot(lstm_pred_prices, label='LSTM Predicted', color='blue', alpha=0.7)
axes[0].set_xlabel('Time Steps')
axes[0].set_ylabel('Price (TWD)')
axes[0].set_title('LSTM: True vs Predicted Price')
axes[0].legend()
axes[0].grid(True)

# Transformer
axes[1].plot(transformer_true_prices, label='True Price', color='black', linewidth=2)
axes[1].plot(transformer_pred_prices, label='Transformer Predicted', color='red', alpha=0.7)
axes[1].set_xlabel('Time Steps')
axes[1].set_ylabel('Price (TWD)')
axes[1].set_title('Transformer: True vs Predicted Price')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('predictions_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# 9.3 Attention Heatmap
def plot_attention_heatmap(model, test_loader, device):
    """繪製 Attention Heatmap"""
    model.eval()

    # 取第一個測試樣本
    X_sample, _ = next(iter(test_loader))
    X_sample = X_sample[0:1].to(device)  # 只取一個樣本

    # 取得注意力權重
    with torch.no_grad():
        attentions = model.get_attention_weights(X_sample)

    # 繪製最後一層的注意力
    last_layer_attn = attentions[-1][0].numpy()  # (seq_len, seq_len)

    plt.figure(figsize=(10, 8))
    sns.heatmap(last_layer_attn, cmap='viridis', cbar=True)
    plt.xlabel('Key Position')
    plt.ylabel('Query Position')
    plt.title('Transformer Last Layer Attention Heatmap')
    plt.savefig('attention_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_attention_heatmap(transformer_model, test_loader, device)

# ============================================================================
# 10. 技術指標比較實驗
# ============================================================================

def compare_indicator_groups(df, lookback=30):
    """比較四類技術指標的效果"""

    # 定義四類指標
    indicator_groups = {
        '趨勢型': ['SMA_5', 'SMA_20', 'EMA_12', 'EMA_26', 'MACD', 'MACD_SIGNAL', 'MACD_HIST'],
        '震盪型': ['RSI_14', 'STOCH_K', 'STOCH_D'],
        '波動型': ['BB_MIDDLE', 'BB_UPPER', 'BB_LOWER', 'BB_WIDTH', 'ATR_14'],
        '量能型': ['Volume', 'OBV', 'VOLUME_SMA_20', 'VOLUME_RATIO']
    }

    base_features = ['Open', 'High', 'Low', 'Close']
    results = {}

    for group_name, indicators in indicator_groups.items():
        print(f"\n測試 {group_name} 指標...")

        # 選擇特徵
        selected_features = base_features + indicators
        available_features = [f for f in selected_features if f in df.columns]

        # 準備資料
        temp_df = df[available_features + ['logret']].copy()
        temp_df['target'] = temp_df['logret'].shift(-1)
        temp_df = temp_df.dropna()

        # 時間序切分
        n = len(temp_df)
        train_size = int(n * 0.8)
        val_size = int(n * 0.1)

        train_df = temp_df.iloc[:train_size]
        test_df = temp_df.iloc[train_size+val_size:]

        # 標準化
        scaler_X = StandardScaler()
        scaler_y = StandardScaler()

        X_train = scaler_X.fit_transform(train_df[available_features])
        y_train = scaler_y.fit_transform(train_df[['target']])

        X_test = scaler_X.transform(test_df[available_features])
        y_test = scaler_y.transform(test_df[['target']])

        # 建立序列
        def create_sequences(X, y, lookback):
            Xs, ys = [], []
            for i in range(len(X) - lookback):
                Xs.append(X[i:i+lookback])
                ys.append(y[i+lookback])
            return np.array(Xs), np.array(ys)

        X_train_seq, y_train_seq = create_sequences(X_train, y_train, lookback)
        X_test_seq, y_test_seq = create_sequences(X_test, y_test, lookback)

        # 建立 DataLoader
        train_dataset = StockDataset(X_train_seq, y_train_seq)
        test_dataset = StockDataset(X_test_seq, y_test_seq)

        train_loader = DataLoader(train_dataset, batch_size=64, shuffle=False)
        test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

        # 訓練簡單的 Transformer
        input_size = X_train_seq.shape[2]
        model = TransformerModel(
            input_size=input_size, d_model=64, nhead=4,
            num_layers=1, dim_feedforward=128, dropout=0.1
        ).to(device)

        criterion = nn.MSELoss()
        optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

        # 快速訓練 15 epochs
        for epoch in range(15):
            train_epoch(model, train_loader, criterion, optimizer, device)

        # 評估
        _, test_preds, test_actuals = evaluate(model, test_loader, criterion, device)

        # 計算指標
        rmse_norm = np.sqrt(mean_squared_error(test_actuals, test_preds))
        mae_norm = mean_absolute_error(test_actuals, test_preds)

        y_true_orig = scaler_y.inverse_transform(test_actuals)
        y_pred_orig = scaler_y.inverse_transform(test_preds)
        rmse_orig = np.sqrt(mean_squared_error(y_true_orig, y_pred_orig))
        mae_orig = mean_absolute_error(y_true_orig, y_pred_orig)

        results[group_name] = {
            'RMSE (Normalized)': rmse_norm,
            'MAE (Normalized)': mae_norm,
            'RMSE (Original)': rmse_orig,
            'MAE (Original)': mae_orig
        }

        print(f"{group_name} - RMSE: {rmse_norm:.6f}, MAE: {mae_norm:.6f}")

    return results

# 執行技術指標比較
indicator_results = compare_indicator_groups(df, lookback=30)

# 視覺化比較結果
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

groups = list(indicator_results.keys())
rmse_values = [indicator_results[g]['RMSE (Normalized)'] for g in groups]
mae_values = [indicator_results[g]['MAE (Normalized)'] for g in groups]

axes[0].bar(groups, rmse_values, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
axes[0].set_ylabel('RMSE (Normalized)')
axes[0].set_title('技術指標類別比較 - RMSE')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(groups, mae_values, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
axes[1].set_ylabel('MAE (Normalized)')
axes[1].set_title('技術指標類別比較 - MAE')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('indicator_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# ============================================================================
# 11. 結果總結
# ============================================================================

print("\n" + "="*80)
print("實驗結果總結")
print("="*80)

print("\n【模型比較】")
print(f"LSTM      - RMSE (Normalized): {lstm_metrics['rmse_norm']:.6f}, "
      f"RMSE (Original): {lstm_metrics['rmse_orig']:.6f}")
print(f"Transformer - RMSE (Normalized): {transformer_metrics['rmse_norm']:.6f}, "
      f"RMSE (Original): {transformer_metrics['rmse_orig']:.6f}")

print("\n【技術指標比較】")
for group_name, metrics in indicator_results.items():
    print(f"{group_name:8s} - RMSE: {metrics['RMSE (Normalized)']:.6f}, "
          f"MAE: {metrics['MAE (Normalized)']:.6f}")

print("\n【資料切分說明】")
print("- 採用時間序列切分 (8:1:1)")
print("- 訓練集用於 fit StandardScaler")
print("- 驗證集用於模型選擇")
print("- 測試集完全未見過,避免資料洩漏")

print("\n【目標變數說明】")
print("- 使用 log return = log(Close[t+1]) - log(Close[t])")
print("- 優點: 降低價格位階偏移,更穩定的分佈")
print("- 可透過累積 log return 還原回價格")

print("\n所有圖表已儲存:")
print("- learning_curves.png")
print("- predictions_comparison.png")
print("- attention_heatmap.png")
print("- indicator_comparison.png")

print("\n程式執行完成!")
